In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from features.category_probability_extractor import CategoryProbabilityExtractor
from features.brand_imputer import BrandImputer
from features.pack_size_probability_extractor import PackSizeProbabilityExtractor
from features.data_preprocessor import DataPreprocessor
from features.cross_validation_ensemble import CrossValidationEnsemble
from sklearn.ensemble import HistGradientBoostingClassifier

In [3]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

category probability

In [4]:
base_lr = LogisticRegression(max_iter=300, random_state=42)
extractor = CategoryProbabilityExtractor(base_model=base_lr, n_splits=5)
extractor.fit(train)

              precision    recall  f1-score   support

       audio       0.90      0.96      0.93      2155
      beauty       1.00      0.97      0.98      1364
       books       0.92      0.99      0.95      1462
   computers       0.97      0.97      0.97      2318
      garden       0.97      0.96      0.97      1050
     kitchen       0.95      0.95      0.95      1864
       music       0.94      0.55      0.69       466
        pets       1.00      0.97      0.99       714
       sport       0.94      0.93      0.94      1580

    accuracy                           0.95     12973
   macro avg       0.95      0.92      0.93     12973
weighted avg       0.95      0.95      0.95     12973



In [5]:
train = extractor.transform(train)
train['declared_category'] = train['declared_category'].fillna('missing')

test = extractor.transform(test)
test['declared_category'] = test['declared_category'].fillna('missing')

brands

In [6]:
bi = BrandImputer(['Noro', 'Velto', 'Aster', 'Mira', 'Tenso'])
train = bi.transform(train)
test = bi.transform(test)

In [7]:
model = LogisticRegression(C=10.0, max_iter=1000)
ppe = PackSizeProbabilityExtractor(model)
ppe.fit(train)
train = ppe.transform(train)
test = ppe.transform(test)

              precision    recall  f1-score   support

         1.0       1.00      1.00      1.00      9329
         2.0       1.00      1.00      1.00      2038
         3.0       1.00      1.00      1.00      1000
         5.0       1.00      1.00      1.00       515

    accuracy                           1.00     12882
   macro avg       1.00      1.00      1.00     12882
weighted avg       1.00      1.00      1.00     12882



In [8]:
train

,id,seller_id,title,description,declared_category,brand,condition,price_pln,shipping_weight_kg,pack_size,target,declared_cat_prob,predicted_category,declared_pack_size_prob,predicted_pack_size
0,OFR_191714271,SEL_7523257,recording mic support — pack of one — Mira A513,Pack of one main items. Microphone stand by Mi...,audio,Mira,used,70.43,5.233,1.0,1,0.812076,audio,0.998570,1.0
1,OFR_517640949,SEL_2053194,in-ear cable headphones — quantity 2 — Noro E315,Included main items: 2. Wired earbuds by Noro....,computers,Noro,used,28.95,0.119,2.0,1,0.008879,audio,0.966432,2.0
2,OFR_634805877,SEL_8620752,Tenso digital food balance / standard / quanti...,Included main items: 1. Kitchen scale by Tenso...,kitchen,Tenso,new,62.05,0.412,1.0,0,0.984746,kitchen,0.988274,1.0
3,OFR_643990737,SEL_8396386,litter scoop — five main items — Velo ...,Cat tray cleaning shovel by Velto. Variant: st...,pets,Velto,used,102.26,0.499,5.0,0,0.922348,pets,0.958346,5.0
4,OFR_260742748,SEL_5029882,LAPTOP SLEEVE — PACK OF FIVE — ASTER F219,Variant: standard. Also includes 3 pcs of free...,computers,Aster,used,144.84,0.856,5.0,0,0.929330,computers,0.985769,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15995,OFR_938431337,SEL_99230,Aster planting hand protection / standard / tw...,Gardening gloves by Aster. Variant: standard. ...,garden,Aster,new,35.23,0.230,2.0,0,0.972309,garden,0.994236,2.0
15996,OFR_920123932,SEL_4497749,soap bar — 5 pcs — Tenso D800,NaN,beauty,Tenso,new,67.23,1.918,5.0,1,0.549050,beauty,0.955879,5.0
15997,OFR_704033898,SEL_2730346,Aster swimming goggles / standard / 3-pack,Pool eye protection by Aster. Variant: standar...,sport,Aster,new,202.82,NaN,3.0,0,0.968398,sport,0.981680,3.0
15998,OFR_442461332,SEL_7926600,graphics tablet — quantity 1 — Noro D130,Variant: large. Also includes 1 pcs of free ac...,computers,Noro,used,1080.99,0.701,1.0,1,0.964732,computers,0.994007,1.0


In [9]:
train["product_type"] = train["description"].fillna("").apply(
    lambda x: x.split(" by ")[0].split(".")[-1].strip().lower() if " by " in x else "missing"
)
test["product_type"] = test["description"].fillna("").apply(
    lambda x: x.split(" by ")[0].split(".")[-1].strip().lower() if " by " in x else "missing"
)

CATEGORICAL_FEATURES = [
    'declared_category', 
    'predicted_category', 
    'brand', 
    'condition', 
    'product_type'
]

NUMERICAL_FEATURES = [
    'declared_cat_prob',
    'declared_pack_size_prob',
    'category_mismatch',
    'pack_size_mismatch',
    'z_score_price',
    'z_score_weight',
    'log_unit_price',
    'log_unit_weight',
    'price_per_weight'
]

TEXT_FEATURES = ['title', 'description']
FEATURES = CATEGORICAL_FEATURES + NUMERICAL_FEATURES + TEXT_FEATURES



In [10]:
from catboost import CatBoostClassifier

dp = DataPreprocessor()
base_model_cb = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3,
    cat_features=CATEGORICAL_FEATURES,
    text_features=TEXT_FEATURES,
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=250,
    allow_writing_files=False,
)
cve = CrossValidationEnsemble(
    base_model=base_model_cb,
    preprocessor=dp,
    features=FEATURES,
    n_splits=5
)
cve.fit(train)


--- Fold 1 ---
0:	learn: 0.6786440	total: 266ms	remaining: 6m 38s
250:	learn: 0.3959700	total: 29.7s	remaining: 2m 28s
500:	learn: 0.2902873	total: 59.6s	remaining: 1m 58s
750:	learn: 0.2372900	total: 1m 26s	remaining: 1m 26s
1000:	learn: 0.2014616	total: 1m 53s	remaining: 56.7s
1250:	learn: 0.1752959	total: 2m 20s	remaining: 28s
1499:	learn: 0.1537843	total: 2m 49s	remaining: 0us
ROC-AUC: 0.9541 | PR-AUC: 0.9063
--- Fold 2 ---
0:	learn: 0.6803165	total: 98ms	remaining: 2m 26s
250:	learn: 0.3801891	total: 28.7s	remaining: 2m 22s
500:	learn: 0.2822688	total: 1m	remaining: 2m
750:	learn: 0.2295523	total: 1m 29s	remaining: 1m 29s
1000:	learn: 0.1969452	total: 1m 57s	remaining: 58.8s
1250:	learn: 0.1714090	total: 2m 25s	remaining: 29s
1499:	learn: 0.1521536	total: 2m 53s	remaining: 0us
ROC-AUC: 0.9435 | PR-AUC: 0.8874
--- Fold 3 ---
0:	learn: 0.6785135	total: 92.7ms	remaining: 2m 19s
250:	learn: 0.3936301	total: 27.2s	remaining: 2m 15s
500:	learn: 0.2880636	total: 54.7s	remaining: 1m 49s
7

In [11]:
test_preds = cve.predict_proba(test)

In [ ]:
submission = test[['id']].copy()
submission['target'] = test_preds
submission.to_csv('submission.csv', index=False)

In [13]:
import hashlib
import time
from catboost import CatBoostClassifier
from features.cross_validation_ensemble import CrossValidationEnsemble
from features.data_preprocessor import DataPreprocessor
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

SEED = 42
fold_assignment = np.array([
    int(hashlib.sha256(f'{SEED}:{s}'.encode('utf-8')).hexdigest()[:16], 16) % 5
    for s in train['seller_id'].astype(str)
])

# --- Zestawy cech do porównania ---
cat_full = [
    'declared_category',
    'predicted_category',
    'brand',
    'condition',
    'product_type',
]
num_full = [
    'declared_cat_prob',
    'declared_pack_size_prob',
    'category_mismatch',
    'pack_size_mismatch',
    'z_score_price',
    'z_score_weight',
    'log_unit_price',
    'log_unit_weight',
    'price_per_weight',
]
text_full = ['title', 'description']
features_full = cat_full + num_full + text_full

# Eksperyment A: Model BEZ 'product_type'
cat_no_pt = [c for c in cat_full if c != 'product_type']
features_no_pt = cat_no_pt + num_full + text_full

# Eksperyment B: Model BEZ cech wagowo-cenowych per sztuka i z-score
num_no_price_weight = [
    c
    for c in num_full
    if c
    not in [
        'z_score_price',
        'z_score_weight',
        'log_unit_price',
        'log_unit_weight',
        'price_per_weight',
    ]
]
features_no_pw = cat_full + num_no_price_weight + text_full

experiments = {
    'Full Model (Referencyjny)': {
        'features': features_full,
        'cat_features': cat_full,
    },
    'Eksperyment A: Bez product_type': {
        'features': features_no_pt,
        'cat_features': cat_no_pt,
    },
    'Eksperyment B: Bez cech cen/wag (unit & z-score)': {
        'features': features_no_pw,
        'cat_features': cat_full,
    },
}


results = []

for exp_name, exp_cfg in experiments.items():
  print('\n' + '=' * 50)
  print(f'{exp_name}')
  print(f'Liczba cech: {len(exp_cfg["features"])}')
  print('=' * 50)

  cb_params = {
      'iterations': 1500,
      'learning_rate': 0.03,
      'depth': 6,
      'l2_leaf_reg': 3,
      'cat_features': exp_cfg['cat_features'],
      'text_features': text_full,
      'auto_class_weights': 'Balanced',
      'random_seed': SEED,
      'verbose': 0,  
      'allow_writing_files': False,
  }

  model_cb = CatBoostClassifier(**cb_params)
  dp = DataPreprocessor()

  cve = CrossValidationEnsemble(
      base_model=model_cb,
      preprocessor=dp,
      features=exp_cfg['features'],
      n_splits=5,
  )

  start_time = time.perf_counter()
  cve.fit(train)
  elapsed_sec = time.perf_counter() - start_time

  
  fold_scores = []
  for f in range(5):
    val_idx = np.flatnonzero(fold_assignment == f)
    prep = cve.preprocessors_[f]
    model = cve.models_[f]

    val_trans = prep.transform(train.iloc[val_idx])
    preds = model.predict_proba(val_trans[exp_cfg['features']])[:, 1]
    fold_scores.append(roc_auc_score(train.iloc[val_idx]['target'], preds))

  mean_auc = float(np.mean(fold_scores))

  print(
      f'Ukończono w {int(elapsed_sec // 60)}m {int(elapsed_sec % 60)}s | Mean'
      f' ROC-AUC: {mean_auc:.4f}'
  )

  results.append({
      'Eksperyment': exp_name,
      'Fold 1': round(fold_scores[0], 4),
      'Fold 2': round(fold_scores[1], 4),
      'Fold 3': round(fold_scores[2], 4),
      'Fold 4': round(fold_scores[3], 4),
      'Fold 5': round(fold_scores[4], 4),
      'Mean ROC-AUC': round(mean_auc, 4),
      'Runtime': f'{int(elapsed_sec // 60)}m {int(elapsed_sec % 60)}s',
      'raw_mean': mean_auc,
  })


ablation_df = pd.DataFrame(results)
baseline_auc = ablation_df.loc[
    ablation_df['Eksperyment'] == 'Full Model (Referencyjny)', 'raw_mean'
].values[0]

ablation_df['Δ (vs Full)'] = (ablation_df['raw_mean'] - baseline_auc).apply(
    lambda x: f'{x:+.4f}'
)
ablation_df = ablation_df.drop(columns=['raw_mean'])

display(ablation_df)


Full Model (Referencyjny)
Liczba cech: 16
--- Fold 1 ---
ROC-AUC: 0.9541 | PR-AUC: 0.9063
--- Fold 2 ---
ROC-AUC: 0.9435 | PR-AUC: 0.8874
--- Fold 3 ---
ROC-AUC: 0.9622 | PR-AUC: 0.9215
--- Fold 4 ---
ROC-AUC: 0.9544 | PR-AUC: 0.9148
--- Fold 5 ---
ROC-AUC: 0.9531 | PR-AUC: 0.8890

OOF ROC-AUC : 0.9537
OOF PR-AUC  : 0.9035
Ukończono w 16m 15s | Mean ROC-AUC: 0.9535

Eksperyment A: Bez product_type
Liczba cech: 15
--- Fold 1 ---
ROC-AUC: 0.9229 | PR-AUC: 0.8578
--- Fold 2 ---
ROC-AUC: 0.9156 | PR-AUC: 0.8439
--- Fold 3 ---
ROC-AUC: 0.9363 | PR-AUC: 0.8814
--- Fold 4 ---
ROC-AUC: 0.9278 | PR-AUC: 0.8749
--- Fold 5 ---
ROC-AUC: 0.9293 | PR-AUC: 0.8556

OOF ROC-AUC : 0.9266
OOF PR-AUC  : 0.8626
Ukończono w 18m 53s | Mean ROC-AUC: 0.9264

Eksperyment B: Bez cech cen/wag (unit & z-score)
Liczba cech: 11
--- Fold 1 ---
ROC-AUC: 0.7315 | PR-AUC: 0.6045
--- Fold 2 ---
ROC-AUC: 0.7219 | PR-AUC: 0.5970
--- Fold 3 ---
ROC-AUC: 0.7439 | PR-AUC: 0.6297
--- Fold 4 ---
ROC-AUC: 0.7364 | PR-AUC: 0.615

,Eksperyment,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean ROC-AUC,Runtime,Δ (vs Full)
0,Full Model (Referencyjny),0.9541,0.9435,0.9622,0.9544,0.9531,0.9535,16m 15s,+0.0000
1,Eksperyment A: Bez product_type,0.9229,0.9156,0.9363,0.9278,0.9293,0.9264,18m 53s,-0.0271
2,Eksperyment B: Bez cech cen/wag (unit & z-score),0.7315,0.7219,0.7439,0.7364,0.7370,0.7341,18m 48s,-0.2193


In [ ]:

import hashlib
import numpy as np
import pandas as pd


fold_assignment = np.array([
    int(hashlib.sha256(f"42:{s}".encode("utf-8")).hexdigest()[:16], 16) % 5
    for s in train["seller_id"].astype(str)
])


oof_preds = np.zeros(len(train))
for f in range(5):
    val_idx = np.flatnonzero(fold_assignment == f)
    prep = cve.preprocessors_[f]
    model = cve.models_[f]
    val_trans = prep.transform(train.iloc[val_idx])
    oof_preds[val_idx] = model.predict_proba(val_trans[cve.features])[:, 1]


oof_df = train.copy()
oof_df["oof_prob"] = np.round(oof_preds, 4)
oof_df["fold"] = fold_assignment + 1

fp = oof_df[oof_df["target"] == 0].nlargest(3, "oof_prob")[
    ["id", "fold", "target", "oof_prob", "title", "description", "declared_category", "price_pln", "shipping_weight_kg", "pack_size"]
]


fn = oof_df[oof_df["target"] == 1].nsmallest(2, "oof_prob")[
    ["id", "fold", "target", "oof_prob", "title", "description", "declared_category", "price_pln", "shipping_weight_kg", "pack_size"]
]

print("=== 3 FALSE POSITIVES ===")
display(fp)

print("\n=== 2 FALSE NEGATIVES ===")
display(fn)

=== 3 FALSE POSITIVES ===


,id,fold,target,oof_prob,title,description,declared_category,price_pln,shipping_weight_kg,pack_size
9023,OFR_537731718,5,0,0.9852,NaN,Guitar strings by Velto. Variant: standard. Pa...,music,NaN,0.277,5.0
10232,OFR_806144714,5,0,0.9844,hair cleansing wash — pack of fiv — Tenso D589,NaN,beauty,151.43,1.756,5.0
11383,OFR_315719964,2,0,0.9797,MIRA NAIL FILE / LARGE /3 UNITS,Bundle quantity: 3 units. Manicure shaping too...,beauty,57.05,0.147,3.0



=== 2 FALSE NEGATIVES ===


,id,fold,target,oof_prob,title,description,declared_category,price_pln,shipping_weight_kg,pack_size
15360,OFR_275857973,1,1,0.0852,LAPTOP VENTILATION STAND — PACK OF ONE — TENSO...,Pack of one main items. Cooling pad by Tenso. ...,computers,85.61,0.474,1.0
1960,OFR_196691591,4,1,0.0976,city visitor handbook — 1 units — Tenso...,Bundle quantity: 1 units. Travel guide by Tens...,books,36.76,0.177,1.0
